In [14]:
import sys
sys.path.insert(0, '/home/jazz/Projects/Statistical-Learning-e20452')

import pandas as pd
import numpy as np
from itables import init_notebook_mode
from open_dataset_store import quick_start
import plotly.express as px
store = quick_start('./ExperimentResults', backend='local')

init_notebook_mode(all_interactive=True)

# import itables.options as opt
# opt.lengthMenu = [10, 25, 50]
# opt.scrollX = True

CSV_PATH = './results/state_log.csv'

Store initialised at: ./ExperimentResults (Backend: local)


In [15]:
df = pd.read_csv(CSV_PATH)
df = df.copy()
base_year = 2014
def _ep_to_datetime(row):
    day, hour, minute = int(row['DayOfYear']), int(row['Hour']), int(row['Minute'])
    if hour >= 24:
        day += 1
        hour -= 24
    return pd.Timestamp(year=base_year, month=1, day=1) + pd.Timedelta(days=day-1, hours=hour, minutes=minute)

df['Datetime'] = df.apply(_ep_to_datetime, axis=1)
df['timestamp'] = df['Datetime'].astype('int64') // 10**9
ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
print(f'Loaded {len(df)} timesteps, columns: {len(df.columns)}')

# list(df.columns)
# df.head(n=20)
# sum = store.get_df_summary(df, detailed=True)


Loaded 768 timesteps, columns: 101


In [16]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------
# 1. AHU Monitoring (Updated with Setpoints, Fan Flows, & Power)
# ---------------------------------------------------------
def plot_ahu_monitoring(df):
    """
    Creates subplots to monitor Temperature, Humidity, Air Flow, CO2, and Power 
    across the Air Handling Unit, including setpoints.
    """
    stages = ['Outdoor_Air', 'Relief_Air', 'Mixer_Inlet', 'Mixed_Air', 'CC_Out', 'HC_Out', 'Fan_Out']
    
    # Added a 5th row for Power
    fig = make_subplots(
        rows=5, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.04,
        subplot_titles=('Temperatures & Setpoints (°C)', 'Relative Humidity (%)', 'Air Flow & Setpoints (kg/s)', 'CO2 Levels (ppm)', 'AHU Power Consumption (W)')
    )

    # 1. Temperature (Including CC and HC Setpoints)
    for stage in stages:
        col = f'{stage}_Temp_C'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=1, col=1)
            
    # Add Temperature Setpoints as dashed lines
    for col in ['Act_CC_Temp_SP_C', 'Act_HC_Temp_SP_C']:
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines', line=dict(dash='dash')), row=1, col=1)

    # 2. Humidity
    for stage in stages:
        col = f'{stage}_RH_pct'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=2, col=1)

    # 3. Flow (Including Fan Flow and OA Setpoints)
    for stage in stages:
        col = f'{stage}_Flow_kg_s'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=3, col=1)
            
    # Add Air Flow Setpoints as dashed lines
    for col in ['Act_Fan_Flow_kg_s', 'Act_OA_Flow_SP_kg_s']:
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines', line=dict(dash='dash')), row=3, col=1)

    # 4. CO2
    for stage in stages:
        col = f'{stage}_CO2_ppm'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=4, col=1)
            
    # 5. Power (CC, HC, Fan Power)
    for col in ['CC_Power_W', 'HC_Power_W', 'Fan_Power_W']:
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=5, col=1)

    fig.update_layout(height=1200, title_text="AHU System Monitoring (Incl. Setpoints & Power)", hovermode="x unified")
    fig.show()


# ---------------------------------------------------------
# 2. Zone Monitoring (Updated with Reheat & Flow Setpoints)
# ---------------------------------------------------------
def plot_zone_monitoring(df, zone_name="SPACE5-1"):
    """
    Plots all relevant metrics for a specifically chosen zone.
    Example zone_names: 'SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1'
    """
    # Extract columns that belong to the selected zone
    zone_cols = [col for col in df.columns if zone_name in col]
    
    # Group them by metric type (Now capturing Reheat Setpoints and Flow Setpoints)
    temp_cols = [c for c in zone_cols if 'Temp_C' in c or 'T_m_C' in c or 'Reheat_SP_C' in c]
    rh_cols = [c for c in zone_cols if 'RH_pct' in c]
    flow_cols = [c for c in zone_cols if 'Flow' in c]
    co2_cols = [c for c in zone_cols if 'CO2' in c]
    power_cols = [c for c in zone_cols if 'Load_W' in c or 'Reheater_W' in c]
    occupant_cols = [c for c in zone_cols if 'Occupants' in c]
    
    fig = make_subplots(
        rows=5, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.04,
        subplot_titles=(
            f'{zone_name} Temperatures & Reheat SP', 
            f'{zone_name} Humidity & CO2', 
            f'{zone_name} Air Flow & Flow SP', 
            f'{zone_name} Power (Reheat & Equip)',
            f'{zone_name} Occupancy'
        )
    )

    # 1. Temp (Dash the setpoint line)
    for col in temp_cols:
        line_style = dict(dash='dash') if 'SP' in col else dict(dash='solid')
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, line=line_style), row=1, col=1)
        
    # 2. RH & CO2
    for col in rh_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col), row=2, col=1)
    for col in co2_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, line=dict(dash='dot')), row=2, col=1)
        
    # 3. Flow (Dash the setpoint line)
    for col in flow_cols:
        line_style = dict(dash='dash') if 'SP' in col else dict(dash='solid')
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, line=line_style), row=3, col=1)
        
    # 4. Power
    for col in power_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col), row=4, col=1)
        
    # 5. Occupants
    for col in occupant_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, fill='tozeroy'), row=5, col=1)

    fig.update_layout(height=1200, title_text=f"Comprehensive Log for {zone_name}", hovermode="x unified")
    fig.show()

# ---------------------------------------------------------
# 3. Energy Consumption (Unchanged - ready for use)
# ---------------------------------------------------------
def plot_energy_consumption(df):
    energy_cols = ['Meter_Bldg_Elec_J', 'Meter_HVAC_Elec_J', 'Meter_AHU_Elec_J', 'Meter_Bldg_Gas_J']
    
    fig = make_subplots(
        rows=2, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.1,
        subplot_titles=('Interval Energy Consumption (Joules)', 'Cumulative Energy Consumption (Joules)')
    )

    for col in energy_cols:
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=1, col=1)

    for col in energy_cols:
        if col in df.columns:
            cumulative_series = df[col].cumsum()
            fig.add_trace(go.Scatter(x=df['timestamp'], y=cumulative_series, name=f'{col} (Cumulative)', mode='lines'), row=2, col=1)

    fig.update_layout(height=700, title_text="Building Energy Meters", hovermode="x unified")
    fig.show()

In [17]:
plot_ahu_monitoring(df)

In [18]:
plot_zone_monitoring(df, zone_name="SPACE1-1")

In [19]:
plot_energy_consumption(df)